# RAG for Image Captioning - Technical Report

## Project overview

This project builds a retrieval-augmented generation (RAG) pipeline for image captioning using the MS COCO dataset. The system combines image-level metadata, caption text, and retrieval indexing to support caption generation and evaluation.

The main goal is not only to generate captions, but also to preserve a clear distinction between:
- raw language data from COCO captions
- cleaned caption data produced by the preprocessing pipeline
- visual metadata generated from YOLO-World detection results
- final validation checks before using the retrieval system

This distinction is important because visual statistics and language statistics must be analyzed from different data sources.

## 1. Problem definition

Image captioning models may generate captions that are syntactically fluent but semantically inaccurate. Typical issues include:
- missing objects in the image
- hallucinated objects or attributes
- weak modeling of object relationships and scene context
- poor grounding with the visual structure of the image

The project addresses this by building a retrieval-based augmentation setup. Before generation, the model can retrieve similar image-caption examples, which provide additional scene information and reduce hallucination risk.

## 2. Dataset and source files

The project uses Microsoft COCO 2017. The main dataset is organized under the `data/` folder.

### Input dataset files
- `data/annotations_trainval2017/annotations/captions_train2017.json`
- `data/annotations_trainval2017/annotations/captions_val2017.json`
- `data/annotations_trainval2017/annotations/instances_train2017.json`
- `data/annotations_trainval2017/annotations/instances_val2017.json`
- `data/val2017/` for validation images
- `data/train2017_subset/` for the downloaded training subset

### Split files produced by the project
- `data/coco_rag_data/train_kb_ids.json`
- `data/coco_rag_data/val_ids.json`
- `data/coco_rag_data/test_ids.json`

These split files are the source of truth for anti-leakage and dataset partitioning. They ensure that the knowledge base does not contain validation or test images.

## 3. Data source separation

One of the most important aspects of this project is correctly separating the data sources used for statistics.

### 3.1 Caption corpus
The caption corpus comes from COCO captions and represents the language side of the dataset.
- raw captions are used for original caption statistics
- cleaned captions are used for preprocessing comparisons

### 3.2 Preprocessed caption store
The cleaned caption corpus is generated by `scripts/caption_pipeline.py` and stored as:
- `data/coco_rag_data/captions_kb.json`
- `data/coco_rag_data/captions_kb_flat.jsonl`

Each entry contains:
- `caption_raw`
- `caption_clean`

This is the correct source for caption preprocessing statistics, vocabulary comparison, and caption-length analysis after cleaning.

### 3.3 Unified visual lookup
Visual statistics should come from the unified lookup, not from raw COCO instance annotation files. The lookup is created in `scripts/build_unified_lookup.py` and stored as:
- `data/coco_rag_data/train_kb_lookup.json`
- `data/coco_rag_data/test_lookup.json`
- `data/coco_rag_data/val_lookup.json`

The lookup stores per-image metadata including:
- `objects`
- `attributes`
- `relationships`
- `scene_context`

These fields are used for object frequency analysis, spatial and visual relationship analysis, and scene complexity.

## 4. Project pipeline

The overall pipeline is organized in the following order:

1. Download the COCO subset and metadata
2. Create split files with anti-leakage constraints
3. Build the unified lookup for visual metadata
4. Clean and normalize captions
5. Extract text embeddings and image embeddings
6. Build FAISS indexes
7. Validate the pipeline and retrieval outputs
8. Run final evaluation

This structure ensures that all retrieval and evaluation stages are reproducible and consistent with the dataset splits.

## 5. Required inputs and generated outputs

### Required inputs
These files are required before running the pipeline:
- `data/annotations_trainval2017/annotations/captions_train2017.json`
- `data/annotations_trainval2017/annotations/captions_val2017.json`
- `data/annotations_trainval2017/annotations/instances_train2017.json`
- `data/annotations_trainval2017/annotations/instances_val2017.json`
- `data/val2017/`
- `data/train2017_subset/`

### Generated outputs
The project generates the following output artifacts:
- `data/coco_rag_data/train_kb_ids.json`
- `data/coco_rag_data/val_ids.json`
- `data/coco_rag_data/test_ids.json`
- `data/coco_rag_data/train_kb_lookup.json`
- `data/coco_rag_data/captions_kb.json`
- `data/coco_rag_data/captions_kb_flat.jsonl`
- `data/coco_rag_data/text_embeddings.npy`
- `data/coco_rag_data/text_index.faiss`
- `data/coco_rag_data/image_embeddings/`
- `data/coco_rag_data/image_index.faiss`
- `data/coco_rag_data/phase8_quality_control_report.json`

These files are the concrete outputs of the pipeline and should be treated as the authoritative artifact set for later analysis.

## 6. File responsibilities by stage

### Stage 1: Data preparation
- `scripts/download_coco_rag_subset.py` — downloads the selected COCO subset
- `scripts/create_splits.py` — creates split files
- `scripts/diagnose_images.py` — checks for missing or corrupted images

### Stage 2: Metadata and lookup construction
- `scripts/build_unified_lookup.py` — builds image-level metadata and object/relationship dictionaries
- `data/coco_rag_data/train_kb_lookup.json` — used for scene statistics and object counts

### Stage 3: Caption preprocessing
- `scripts/caption_pipeline.py` — normalizes raw captions and creates `caption_clean`
- `data/coco_rag_data/captions_kb.json` — cleaned caption store

### Stage 4: Embeddings and retrieval
- `scripts/extract_image_embeddings.py` — image embeddings
- `scripts/extract_text_embeddings.py` — text embeddings
- `scripts/build_faiss_index.py` — vector index creation

### Stage 5: Quality assurance
- `scripts/phase8_quality_control.py` — checks the integrity of the retrieval pipeline
- `data/coco_rag_data/phase8_quality_control_report.json` — final validation report

## 7. Recommended analysis logic

The correct analysis logic is the following:

- Use raw COCO captions for original caption-language statistics
- Use `caption_clean` for preprocessing and cleaned-caption statistics
- Use `lookup.objects` for object frequency and scene complexity analysis
- Use `lookup.relationships` only for visual relationship analysis
- Keep caption-based spatial relations and image-based visual relations in separate sections
- Do not compare raw textual statistics with YOLO-derived detector metadata in the same metric group

This prevents misleading interpretations such as using detector objects as if they were a text vocabulary.

## 8. Quality assurance and validation

The project includes a final quality control stage to ensure the retrieval system is valid before downstream evaluation.

The script `scripts/phase8_quality_control.py` checks:
- anti-leakage between train, validation, and test splits
- consistent image and caption mapping
- text embedding integrity
- FAISS index integrity
- alignment between image lookup data and embedding vectors
- simple self-retrieval sanity tests

This is the final gate before using the knowledge base in retrieval-based caption generation.

## 9. Detector rationale, threshold choice, and source-of-truth rules

The visual metadata stage is implemented in `scripts/build_unified_lookup.py` and uses the YOLO-World checkpoint `yolov8s-world.pt`.

### Why YOLO-World was used

YOLO-World was chosen because the project needs open-vocabulary object detection at scale. It is more efficient than heavier object detectors and is better suited to a large COCO-based retrieval pipeline where many images must be processed without requiring extreme GPU cost. In this pipeline, the detector is not used as a final classifier; it is used to generate metadata that grounds the retrieval system in visual content.

The detector supplies visual information such as:
- `objects`
- `attributes`
- `relationships`
- `scene_context_tags`
- `scene_context_primary`

This metadata is then used to enrich the retrieval knowledge base and to support object- and scene-level analysis.

### Why the threshold matters

The confidence threshold acts as a precision/recall tradeoff. A low threshold keeps more candidate detections, but it also increases false positives and noisy objects. A high threshold improves precision, but can eliminate weak but valid detections. The project therefore uses a moderate threshold so that the lookup captures the salient objects and scene structure without overloading the system with spurious detections.

This matters because the lookup is later used for statistics and retrieval grounding. If the threshold is too low, the lookup becomes noisy; if it is too high, visual evidence becomes too sparse. The selected threshold is thus a practical design choice balancing signal quality and scene coverage.

### Why we do not use raw COCO instances as the final object statistic source

The COCO instance annotation JSON is useful as ground truth for object-level labeling and benchmarking, but it is not the authoritative source for the project’s retrieval statistics. The project’s visual analysis is based on the unified lookup generated by YOLO-World, because this is the metadata that has already been filtered, normalized, and aligned with the retrieval pipeline.

This distinction is essential:
- COCO instance annotations describe the dataset annotation structure.
- the unified lookup describes the actual image metadata used in the retrieval pipeline.
- object statistics from the lookup reflect the detector-driven scene representation used by the system.

Therefore, the statistics should be computed from `train_kb_lookup.json`, `val_lookup.json`, and `test_lookup.json`, not directly from the raw `instances_*.json` files.

### Why raw captions remain separate from generated visual metadata

The project keeps the following categories separate:

1. Raw caption statistics from COCO captions
2. Cleaned caption statistics derived from `caption_clean`
3. Visual statistics derived from `lookup.objects`, `lookup.relationships`, and `scene_context`

This separation is intentional. Captions represent lexical and semantic language patterns, while YOLO outputs represent visual scene evidence. Mixing them into the same statistics section would confuse the analysis and produce misleading conclusions.

### Why relationships are used only in a dedicated visual analysis section

The project extracts visual relationships and scene-context tags, but these are not treated as part of the standard caption vocabulary analysis. Their role is different:

- language analysis asks: what words appear, how often, and how long are the captions?
- relationship analysis asks: what objects co-occur, what spatial relations are present, and how scenes are structured?

This means visual relation statistics belong in a separate analysis section, not inside the standard text statistics section. If they are mixed together, the resulting plots and counts will be hard to interpret and may falsely imply that detector outputs are part of the language vocabulary.

### Why NLTK is used

NLTK is used mainly for language-oriented preprocessing and analysis such as tokenization, stop-word removal, POS-related inspection, and text normalization. It is not a replacement for visual understanding and should not be confused with YOLO-World’s visual relationship extraction. In other words, NLTK supports textual analysis, whereas YOLO metadata supports visual scene reasoning.

---

## 9.5 Expected execution results by stage

Each script in the pipeline is supposed to confirm that the produced data is valid before moving to the next stage. The expected behavior is not just that the file exists, but that the file satisfies the logical checks of the project.

### `scripts/create_splits.py`
Expected result:
- train, validation, and test IDs are disjoint
- no image ID appears in more than one split
- split counts match the configured project policy
- the script should report no data leakage

Interpretation: if overlap exists, the project fails the anti-leakage check and the retrieval evaluation becomes invalid.

### `scripts/build_unified_lookup.py`
Expected result:
- every image in the split has a metadata entry in the lookup JSON
- each entry includes `objects`, `attributes`, `relationships`, and scene-context fields
- the visual lookup contains no missing image IDs
- detected entries are consistent with the image split being processed

Interpretation: this output is the source of truth for the visual scene statistics and should be checked before any graph-based analysis is performed.

### `scripts/caption_pipeline.py`
Expected result:
- every caption entry has a raw caption and a cleaned caption
- `caption_clean` is shorter and normalized compared with the original
- duplicate or invalid entries are removed or flagged
- the output store is aligned with the image IDs in the selected split

Interpretation: this script should confirm that the cleaned caption corpus is valid and reproducible, not just another copy of the original captions.

### `scripts/extract_text_embeddings.py`
Expected result:
- the text embedding matrix is created successfully
- the number of embedding rows matches the number of caption entries
- the mapping file and embedding order are aligned
- no missing IDs appear in the text mapping

Interpretation: this step should prove that the caption corpus has been converted into a valid retrieval-compatible vector space.

### `scripts/extract_image_embeddings.py`
Expected result:
- the image feature matrix is created successfully
- the number of image embeddings matches the number of processed image IDs
- embedding rows are aligned with the corresponding image metadata
- no empty or corrupted image vectors remain

Interpretation: the retrieval system must be able to compare image features and text features in the same latent space.

### `scripts/build_faiss_index.py`
Expected result:
- a FAISS index is created for the text and/or image set
- the index dimension matches the embedding dimension
- the index contains the expected number of vectors
- any retrieval test should return valid IDs from the indexed dataset

Interpretation: the index should be readable and consistent with the embedding metadata; it should not contain mismatched dimensions or missing rows.

### `scripts/phase8_quality_control.py`
Expected result:
- anti-leakage check: no intersection between train, validation, and test sets
- file presence check: all required files exist
- mapping check: image IDs and caption IDs remain aligned
- normalization check: embeddings are valid and length-normalized where required
- FAISS smoke test: a query can retrieve a valid self-match or a near neighbor without errors
- final status: the pipeline is either PASS or specific failing checks are reported clearly

Interpretation: this is the final gate. If the script reports a failure, the retrieval pipeline is not considered reliable for evaluation.

In short, the expected execution of each file should produce evidence that the pipeline is clean, aligned, and safe for retrieval, not just a file on disk.

---

## 10. Data and documentation references

The project repository contains supporting references in the `Docs/` folder. These files are useful for understanding the intended design and execution logic:

- `Docs/UpdatedReadme.md`
- `Docs/caption_pipleline.md`
- `Docs/FAISS_Index.md`

These documents should be read together with the scripts in `scripts/`, because they provide the operational context for the cleaning pipeline, retrieval setup, and indexing logic. The scripts are the source of truth for implementation, while the docs explain the intended workflow and design decisions.

---

## 11. Correct execution order and required artifacts

The execution order is important because later stages depend on earlier outputs:

```bash
# 1. set up the environment
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

# 2. download/prepare dataset files
python scripts/download_coco_rag_subset.py

# 3. validate images
python scripts/diagnose_images.py

# 4. create split files
python scripts/create_splits.py

# 5. build unified visual lookup
python scripts/build_unified_lookup.py

# 6. preprocess captions
python scripts/caption_pipeline.py

# 7. preprocess and prepare images
python scripts/preprocess_images.py
# or batch execution:
python scripts/run_preprocessing_batch.py

# 8. extract embeddings
python scripts/extract_text_embeddings.py
python scripts/extract_image_embeddings.py

# 9. build vector indexes
python scripts/build_faiss_index.py

# 10. validate project integrity
python scripts/phase8_quality_control.py
```

The most important project principle is that every output must be generated from the correct source and used in the correct stage. This prevents data leakage, keeps evaluation fair, and preserves a trustworthy retrieval pipeline.

---

## 12. Recommended statistical structure for the final analysis

For the final report, the statistics should be split into the following groups:

### 12.1 Text statistics
- raw caption length distribution
- cleaned caption length distribution
- vocabulary size before and after cleaning
- word-frequency distribution
- most common terms after preprocessing

### 12.2 Visual statistics
- top detected objects from `lookup.objects`
- object frequency distribution
- scene-context frequency
- relationship distribution from `lookup.relationships`
- spatial relationship summary

### 12.3 Retrieval evaluation
- embedding integrity checks
- FAISS self-retrieval sanity check
- split leakage validation
- file and mapping consistency checks

This structure keeps the analysis aligned with the real source data and ensures that the report is scientifically interpretable.

---

## 13. Final interpretation

The project is a multimodal retrieval framework built around a clear data-source separation. Captions are language data, while the YOLO-World lookup represents visual evidence. The retrieval setup is only trustworthy when the pipeline preserves this distinction, avoids leakage, and validates the resulting artifacts before use.

This is why the project keeps raw and cleaned captions separate, why object statistics come from the unified lookup, and why visual relationships remain in a dedicated analysis section. The methodology is consistent, reproducible, and grounded in the actual file pipeline used by the project.
